# RG3 조건부 순위75: 가스 양상별 전문가 앙상블

88번의 정상 이웃75 조건부 순위10개를 고정. 학습 가스만 두 군집으로 나눠 각 군집 대 정상 RF/ET 2시드 전문가를 학습하고 군집 최대·모델 평균. depth8/leaf2로 작은 센서 조합도 표현. std4 기준은 이전 depth4/leaf4 유지. 군집은 실제 가스 원인 유형 확정이 아니다. 평가 가스 군집을 전역으로 학습하지 않음.

RG3 RH620(가스19/정상601). CN7 미사용. 외부 평가 구간은 정상 이웃·군집·분류·임계값 선택에서 제외. 내부 구간검증 후보 선택. 이미 본 자료 개발 성적이며 새 독립 검증 아님.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 state=d[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
 state['pressure_switch_gap']=d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(state.iloc[normal])
 tr=im.transform(state.iloc[ref]);te=im.transform(state.iloc[q])
 si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][stdcols])
 st=si.transform(d.iloc[ref][stdcols]);sq=si.transform(d.iloc[q][stdcols])
 base_tr=np.c_[tr,st];base_q=np.c_[te,sq]
 norm=base_tr[y[ref]==0];normal_ids=ref[y[ref]==0]
 sc=RobustScaler().fit(norm)
 context_tr=np.clip(sc.transform(base_tr),-8,8);context_q=np.clip(sc.transform(base_q),-8,8)
 context_norm=context_tr[y[ref]==0]
 rank_views=[]
 for neighbors in [75]:
  k=min(neighbors,len(norm)-1)
  ranked=[]
  for vals,contexts,ids in [(base_tr,context_tr,ref),(base_q,context_q,q)]:
   z=np.empty_like(vals)
   for col in range(vals.shape[1]):
    other=np.arange(vals.shape[1])!=col
    nn=NearestNeighbors(n_neighbors=k+1).fit(context_norm[:,other])
    idx=nn.kneighbors(contexts[:,other],return_distance=False)
    for row in range(len(vals)):
     candidates=idx[row][normal_ids[idx[row]]!=ids[row]][:k]
     peers=norm[candidates,col]
     z[row,col]=(1+np.sum(peers<vals[row,col])+.5*np.sum(peers==vals[row,col]))/(len(peers)+2)
   ranked.append(z)
  rank_views.append(tuple(ranked))
 features=[(st,sq)]+rank_views
 scores=[]
 for view,(ft,fq) in enumerate(features):
  vals=[]
  if view==0:
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
     vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  else:
   positive=np.flatnonzero(y[ref]==1);neg=np.flatnonzero(y[ref]==0)
   modes=min(2,len(np.unique(ft[positive],axis=0)))
   labels=KMeans(n_clusters=modes,n_init=10,random_state=42).fit_predict(ft[positive]) if modes>1 else np.zeros(len(positive),int)
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     per_mode=[]
     for mode in np.unique(labels):
      idx=np.r_[neg,positive[labels==mode]]
      m=cls(n_estimators=100,max_depth=8,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2)
      per_mode.append(m.fit(ft[idx],y[ref][idx]).predict_proba(fq)[:,1])
     vals.append(np.max(per_mode,axis=0))
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],np.maximum(*scores)])
 audit.append(dict(train=len(ref),query=len(q),normal_imputer_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','rank75_gas_mode_ensemble','std4_rank75_gas_mode_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_rank75_gas_mode_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))





Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                        model  budget  TP  FP  FN  TN   recall       F1       AP
  inner_recall_first_selected    0.00   3  79  16 522 0.157895 0.059406      NaN
     rank75_gas_mode_ensemble    0.02   0  19  19 582 0.000000 0.000000 0.056313
     rank75_gas_mode_ensemble    0.05   1  35  18 566 0.052632 0.036364 0.056313
     rank75_gas_mode_ensemble    0.10   3  60  16 541 0.157895 0.073171 0.056313
std4_rank75_gas_mode_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.059184
std4_rank75_gas_mode_ensemble    0.05   3  37  16 564 0.157895 0.101695 0.059184
std4_rank75_gas_mode_ensemble    0.10   3  70  16 531 0.157895 0.065217 0.059184
               std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
               std4_reference    0.05   3  33  16 568 0.157895 0.109091 0.061578
               std4_reference    0.10   3  67  16 534 0.157895 0.067416 0.061578
